In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
!ls /content/drive/MyDrive/audio_project/datasets


UrbanSound8K


In [ ]:
!ls /content/drive/MyDrive/audio_project/datasets/UrbanSound8K/audio


In [ ]:
!mkdir -p /content/drive/MyDrive/audio_project/project_dataset/danger
!mkdir -p /content/drive/MyDrive/audio_project/project_dataset/alert
!mkdir -p /content/drive/MyDrive/audio_project/project_dataset/safe

!mkdir -p /content/drive/MyDrive/audio_project/processed_dataset/danger
!mkdir -p /content/drive/MyDrive/audio_project/processed_dataset/alert
!mkdir -p /content/drive/MyDrive/audio_project/processed_dataset/safe


In [ ]:
!ls /content/drive/MyDrive/audio_project


audio		    processed_dataset		  smartear_epoch15_savedmodel
datasets	    project_dataset		  spectrogram_dataset
esc50_auto_sort.py  smartear_epoch15_model.h5	  standardize_audio.py
esc50.csv	    smartear_epoch15_model.keras


In [ ]:
import os
import csv
import shutil
from tqdm import tqdm


In [ ]:
BASE = "/content/drive/MyDrive/audio_project"

ESC_AUDIO = f"{BASE}/audio"
ESC_CSV   = f"{BASE}/esc50.csv"

URBAN_AUDIO = f"{BASE}/datasets/UrbanSound8K/audio"
URBAN_CSV   = f"{BASE}/datasets/UrbanSound8K/metadata/UrbanSound8K.csv"

OUTPUT = f"{BASE}/project_dataset"


In [ ]:
danger_classes = [
    "glass_breaking", "gun_shot", "siren", "smoke_alarm",
    "engine_idling", "jackhammer", "screaming"
]

alert_classes = [
    "dog", "children_playing", "crying_baby",
    "door_knock", "footsteps"
]

safe_classes = [
    "rain", "wind", "thunderstorm",
    "clock_tick", "air_conditioner", "silence"
]


In [ ]:
urban_danger = urban_alert = urban_safe = 0

with open(URBAN_CSV) as f:
    reader = csv.DictReader(f)
    for row in tqdm(reader, desc="Sorting UrbanSound8K"):
        fname = row["slice_file_name"]
        label = row["class"]
        fold  = row["fold"]

        src = f"{URBAN_AUDIO}/fold{fold}/{fname}"
        if not os.path.exists(src):
            continue

        if label in danger_classes:
            dst = f"{OUTPUT}/danger/urban_{fname}"
            urban_danger += 1
        elif label in alert_classes:
            dst = f"{OUTPUT}/alert/urban_{fname}"
            urban_alert += 1
        elif label in safe_classes:
            dst = f"{OUTPUT}/safe/urban_{fname}"
            urban_safe += 1
        else:
            continue

        shutil.copy(src, dst)

print("✅ UrbanSound DONE")
print("Danger:", urban_danger)
print("Alert :", urban_alert)
print("Safe  :", urban_safe)


In [ ]:
 # ===== SMART EAR : SETUP CELL =====

from google.colab import drive
drive.mount('/content/drive')

import os, shutil
import librosa
import soundfile as sf
from tqdm import tqdm

# Base paths
BASE = "/content/drive/MyDrive/audio_project"

PROJECT_DATASET = f"{BASE}/project_dataset"
PROCESSED_DATASET = f"{BASE}/processed_dataset"

# Ensure processed folders
for cls in ["danger", "alert", "safe"]:
    os.makedirs(f"{PROCESSED_DATASET}/{cls}", exist_ok=True)

print("✅ Drive mounted")
print("✅ Paths ready")
print("Project dataset:", os.listdir(PROJECT_DATASET))


In [ ]:
# ===== SMART EAR : AUDIO STANDARDIZATION =====

TARGET_SR = 22050       # standard ML audio SR
DURATION = 4.0          # seconds
MAX_LEN = int(TARGET_SR * DURATION)

def standardize_audio(src_path, dst_path):
    try:
        y, sr = librosa.load(src_path, sr=TARGET_SR, mono=True)

        if len(y) > MAX_LEN:
            y = y[:MAX_LEN]
        else:
            y = librosa.util.fix_length(y, MAX_LEN)

        sf.write(dst_path, y, TARGET_SR)
    except Exception as e:
        pass


counts = {"danger": 0, "alert": 0, "safe": 0}

for cls in ["danger", "alert", "safe"]:
    src_dir = f"{PROJECT_DATASET}/{cls}"
    dst_dir = f"{PROCESSED_DATASET}/{cls}"

    files = [f for f in os.listdir(src_dir) if f.endswith(".wav")]

    for f in tqdm(files, desc=f"Standardizing {cls}"):
        src = f"{src_dir}/{f}"
        dst = f"{dst_dir}/{f}"

        standardize_audio(src, dst)
        counts[cls] += 1


print("\n✅ STANDARDIZATION DONE")
print("Danger:", counts["danger"])
print("Alert :", counts["alert"])
print("Safe  :", counts["safe"])


In [ ]:
# ===== FINAL DATASET CHECK =====

for cls in ["danger", "alert", "safe"]:
    path = f"{PROCESSED_DATASET}/{cls}"
    files = os.listdir(path)
    print(f"{cls.upper()} → {len(files)} files")

print("\nSample files:")
print(os.listdir(f"{PROCESSED_DATASET}/danger")[:5])


In [ ]:
print("Sample danger files:")
print(os.listdir(f"{PROCESSED_DATASET}/danger")[:10])


In [ ]:
!ls /content/drive/MyDrive/audio_project/processed_dataset/danger | head


In [ ]:
import os
from tqdm import tqdm

PROJECT_DATASET = "/content/drive/MyDrive/audio_project/project_dataset"
PROCESSED_DATASET = "/content/drive/MyDrive/audio_project/processed_dataset"

counts = {"danger":0, "alert":0, "safe":0}

for cls in ["danger", "alert", "safe"]:
    src_dir = f"{PROJECT_DATASET}/{cls}"
    dst_dir = f"{PROCESSED_DATASET}/{cls}"
    os.makedirs(dst_dir, exist_ok=True)

    for file in tqdm(os.listdir(src_dir), desc=f"Standardizing {cls}"):
        src = os.path.join(src_dir, file)
        dst = os.path.join(dst_dir, file)

        if not os.path.exists(dst):   # important: avoid double work
            standardize_audio(src, dst)
            counts[cls] += 1

print("\n✅ FINAL STANDARDIZATION DONE")
print("Danger:", counts["danger"])
print("Alert :", counts["alert"])
print("Safe  :", counts["safe"])


In [ ]:
 for cls in ["danger","alert","safe"]:
    files = os.listdir(f"/content/drive/MyDrive/audio_project/processed_dataset/{cls}")
    print(cls.upper(), "→", len(files))
    print(files[:5], "\n")


In [ ]:
import os, shutil
from tqdm import tqdm

URBAN_AUDIO = "/content/drive/MyDrive/audio_project/datasets/UrbanSound8K/audio"
URBAN_META  = "/content/drive/MyDrive/audio_project/datasets/UrbanSound8K/metadata/UrbanSound8K.csv"
OUTPUT = "/content/drive/MyDrive/audio_project/project_dataset"

danger_classes = ["gun_shot","siren","jackhammer","engine_idling"]
alert_classes  = ["dog_bark","car_horn","children_playing"]
safe_classes   = ["air_conditioner","street_music"]

import csv

count = {"danger":0,"alert":0,"safe":0}

with open(URBAN_META) as f:
    reader = csv.DictReader(f)
    for row in tqdm(reader, desc="Copying UrbanSound → project_dataset"):
        label = row["class"]
        fold  = row["fold"]
        fname = row["slice_file_name"]

        src = f"{URBAN_AUDIO}/fold{fold}/{fname}"
        if not os.path.exists(src):
            continue

        if label in danger_classes:
            cls = "danger"
        elif label in alert_classes:
            cls = "alert"
        elif label in safe_classes:
            cls = "safe"
        else:
            continue

        dst = f"{OUTPUT}/{cls}/urban_{fname}"
        if not os.path.exists(dst):
            shutil.copy(src, dst)
            count[cls] += 1

print("\n✅ UrbanSound copied")
print(count)


In [ ]:
for cls in ["danger","alert","safe"]:
    files = os.listdir(f"/content/drive/MyDrive/audio_project/project_dataset/{cls}")
    print(cls.upper(), len(files))
    print(files[:5], "\n")


In [ ]:
import os
import librosa
import soundfile as sf
from tqdm import tqdm

PROJECT_DATASET = "/content/drive/MyDrive/audio_project/project_dataset"
PROCESSED_DATASET = "/content/drive/MyDrive/audio_project/processed_dataset"

TARGET_SR = 16000
DURATION = 4
TARGET_LEN = TARGET_SR * DURATION

os.makedirs(PROCESSED_DATASET, exist_ok=True)

def standardize_audio(src, dst):
    try:
        y, sr = librosa.load(src, sr=TARGET_SR, mono=True)

        # Trim or pad safely (librosa 0.10+)
        if len(y) > TARGET_LEN:
            y = y[:TARGET_LEN]
        else:
            y = librosa.util.fix_length(y, size=TARGET_LEN)

        sf.write(dst, y, TARGET_SR)

    except Exception as e:
        print(f"❌ Failed: {os.path.basename(src)} → {e}")

counts = {"danger":0, "alert":0, "safe":0}

for cls in ["danger", "alert", "safe"]:
    in_dir = f"{PROJECT_DATASET}/{cls}"
    out_dir = f"{PROCESSED_DATASET}/{cls}"
    os.makedirs(out_dir, exist_ok=True)

    files = os.listdir(in_dir)

    for fname in tqdm(files, desc=f"Standardizing {cls}"):
        src = f"{in_dir}/{fname}"
        dst = f"{out_dir}/{fname}"

        if not os.path.exists(dst):
            standardize_audio(src, dst)
            counts[cls] += 1

print("\n✅ FINAL STANDARDIZATION DONE")
print("Danger:", counts["danger"])
print("Alert :", counts["alert"])
print("Safe  :", counts["safe"])


In [ ]:
for c in ["danger","alert","safe"]:
    print(c.upper(), len(os.listdir(f"{PROCESSED_DATASET}/{c}")))


In [ ]:
for c in ["danger","alert","safe"]:
    print(c.upper(), len(os.listdir(f"{PROCESSED_DATASET}/{c}")))


In [ ]:
import os
import numpy as np
import librosa
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.model_selection import train_test_split
from tqdm import tqdm


In [ ]:
BASE_PATH = "/content/drive/MyDrive/audio_project/processed_dataset"

CLASSES = {
    "danger": 0,
    "alert": 1,
    "safe": 2
}

SR = 16000
N_MELS = 128
MAX_TIME_FRAMES = 128   # CNN input width


In [ ]:
def wav_to_mel(path):
    y, sr = librosa.load(path, sr=SR, mono=True)

    mel = librosa.feature.melspectrogram(
        y=y,
        sr=sr,
        n_fft=1024,
        hop_length=512,
        n_mels=N_MELS
    )

    mel_db = librosa.power_to_db(mel, ref=np.max)

    # Fix time dimension
    if mel_db.shape[1] < MAX_TIME_FRAMES:
        pad = MAX_TIME_FRAMES - mel_db.shape[1]
        mel_db = np.pad(mel_db, ((0, 0), (0, pad)))
    else:
        mel_db = mel_db[:, :MAX_TIME_FRAMES]

    return mel_db


In [ ]:
X = []
y = []

for cls, label in CLASSES.items():
    folder = os.path.join(BASE_PATH, cls)
    files = os.listdir(folder)

    for file in tqdm(files, desc=f"Loading {cls}"):
        path = os.path.join(folder, file)
        try:
            mel = wav_to_mel(path)
            X.append(mel)
            y.append(label)
        except Exception as e:
            print("❌", file, e)


In [ ]:
X = np.array(X)
y = np.array(y)

# Add channel dimension for CNN
X = X[..., np.newaxis]

# Normalize
X = (X - X.mean()) / (X.std() + 1e-9)

print("X shape:", X.shape)
print("y distribution:", np.unique(y, return_counts=True))


In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train size:", X_train.shape)
print("Val size:", X_val.shape)


In [1]:
model = models.Sequential([

    layers.Conv2D(32, (3,3), activation='relu', input_shape=(128,128,1)),
    layers.BatchNormalization(),
    layers.MaxPooling2D(),

    layers.Conv2D(64, (3,3), activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling2D(),

    layers.Conv2D(128, (3,3), activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling2D(),

    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.4),

    layers.Dense(3, activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()


NameError: name 'models' is not defined

In [ ]:
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=15,
    batch_size=32
)


In [ ]:
model.save("/content/drive/MyDrive/audio_project/smartear_epoch15_model.keras")
print("✅ Model saved successfully (Epoch 15)")


In [ ]:
model.save("/content/drive/MyDrive/audio_project/smartear_epoch15_model.h5")
print("✅ Model saved as H5")


In [ ]:
model.export("/content/drive/MyDrive/audio_project/smartear_epoch15_savedmodel")
print("✅ SavedModel exported")


In [ ]:
from tensorflow.keras.models import load_model

model = load_model("/content/drive/MyDrive/audio_project/smartear_epoch15_model.keras")
print("✅ Model loaded correctly")


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, classification_report, f1_score
import seaborn as sns


In [ ]:
import tensorflow as tf
import numpy as np
import os

PROCESSED_DIR = "/content/drive/MyDrive/audio_project/processed_dataset"
CLASSES = ["danger", "alert", "safe"]
IMG_SIZE = (128, 128)


In [ ]:
def load_mel(path, label):
    mel = np.load(path.numpy().decode())
    mel = np.expand_dims(mel, -1)  # (H, W, 1)
    return mel, label

def tf_load_mel(path, label):
    mel, label = tf.py_function(
        load_mel,
        [path, label],
        [tf.float32, tf.int32]
    )
    mel.set_shape((128, 128, 1))
    return mel, label


In [ ]:
file_paths = []
labels = []

for i, cls in enumerate(CLASSES):
    cls_path = os.path.join(PROCESSED_DIR, cls)
    for f in os.listdir(cls_path):
        if f.endswith(".npy"):
            file_paths.append(os.path.join(cls_path, f))
            labels.append(i)

file_paths = tf.constant(file_paths)
labels = tf.constant(labels)

val_ds = tf.data.Dataset.from_tensor_slices((file_paths, labels))
val_ds = val_ds.map(tf_load_mel)
val_ds = val_ds.batch(32)


In [ ]:
from sklearn.metrics import confusion_matrix, f1_score, classification_report
import seaborn as sns
import matplotlib.pyplot as plt

y_true, y_pred = [], []

for x, y in val_ds:
    preds = model.predict(x, verbose=0)
    y_true.extend(y.numpy())
    y_pred.extend(np.argmax(preds, axis=1))


In [ ]:
print("y_true length:", len(y_true))
print("y_pred length:", len(y_pred))


In [ ]:
import tensorflow as tf
import numpy as np
import os

DATASET_DIR = "/content/drive/MyDrive/audio_project/processed_dataset"
CLASSES = ["danger", "alert", "safe"]


In [ ]:
file_paths = []
labels = []

for idx, cls in enumerate(CLASSES):
    cls_dir = os.path.join(DATASET_DIR, cls)
    for f in os.listdir(cls_dir):
        if f.endswith(".npy"):
            file_paths.append(os.path.join(cls_dir, f))
            labels.append(idx)

print("Total samples:", len(file_paths))


In [ ]:
import numpy as np
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns
import matplotlib.pyplot as plt

CLASSES = ["danger", "alert", "safe"]

y_true = []
y_pred = []

for x, y in val_ds:
    preds = model.predict(x, verbose=0)
    y_true.extend(np.argmax(y.numpy(), axis=1))
    y_pred.extend(np.argmax(preds, axis=1))

y_true = np.array(y_true)
y_pred = np.array(y_pred)


In [ ]:
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(6,5))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=CLASSES,
    yticklabels=CLASSES
)
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Confusion Matrix")
plt.show()
